# 04 — Bayesian fitting with DREAM (Solution)

**Time:** 60–75 minutes (advanced extension)
**Question:** what can a posterior distribution tell us that a single best fit
cannot?

Earlier notebooks returned one optimum and a local uncertainty estimate. Here
you will sample a distribution of plausible parameter combinations using the
DREAM Markov chain Monte Carlo algorithm.

> **Lecture connection**
>
> - Slide 32, *How can uncertainties help: posteriors, correlations and bounds*:
>   this notebook produces exactly those three objects for the sphere from
>   notebook 01.
> - Slide 34, *Wrap-up for part 2*: "several of these parameters are
>   degenerate ($\phi \leftrightarrow \Delta\rho \leftrightarrow$ scale)". Here
>   you will *see* a degeneracy as a tilted cloud in a pair plot.
> - Slide 18's discussion: concentration and contrast move the curve up and
>   down, while size moves it left and right. Keep that picture in mind when
>   you predict the correlations below.
> - Notebook 02's non-identifiability is the extreme case of what you measure
>   here: a posterior that is long and flat in some direction.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'sphere_reference.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']


## 1 — From bounds to priors

Bayes' theorem combines a likelihood with prior information:

$$
p(\theta \mid D) \propto p(D \mid \theta)\,p(\theta).
$$

For this workshop, `set_param(..., min=a, max=b, vary=True)` defines a
**uniform prior** between `a` and `b`: every value inside is initially equally
plausible and values outside are forbidden. This is not merely an optimizer
safety rail. Bounds must therefore be scientifically defensible.

Configure the same sphere model as before. Keep known SLD values fixed and use
bounded priors for radius, scale, and background.

**Where the likelihood comes from.** If each measured point is Gaussian with
standard deviation `dI`, then
$p(D \\mid \\theta) \\propto \\exp(-\\chi^2(\\theta)/2)$, with the same $\\chi^2$ you
minimized in notebook 01. The best fit is the peak of this function. Sampling
maps out its whole shape. This is why correct error bars from data reduction
(slides 42–43) matter even more here than for a point estimate.

**Fixing the SLDs is itself a prior.** It is a delta-function prior that
breaks the $\\phi \\leftrightarrow \\Delta\\rho$ degeneracy of slide 34 by fiat.
Ask yourself how confident you really are in those numbers (slide 11's
caveats about density and exchangeable hydrogens).

In [ ]:
fitter = SANSFitter()
fitter.load_data(str(filename))
fitter.set_model('sphere')
fitter.set_param('sld', value=4.2, vary=False)
fitter.set_param('sld_solvent', value=1.0, vary=False)
fitter.set_param('radius', value=50, min=30, max=90, vary=True)
fitter.set_param('scale', value=0.05, min=0.01, max=0.12, vary=True)
fitter.set_param('background', value=0.002, min=0, max=0.008, vary=True)
fitter.get_params()


## 2 — Sampling with DREAM

`fit_bayesian` uses several controls:

- `samples`: approximate posterior sampling budget;
- `burn`: early generations discarded while chains approach high probability;
- `thin`: retain every nth draw;
- `pop`: population size multiplier for DREAM's parallel chains.

A larger budget is not automatically trustworthy. Chains must mix, explore the
same region, and be insensitive to starting values. The short run below is
chosen for class time and is **exploratory, not publication quality**.

DREAM is a population method, like the differential-evolution optimizer from
the notebook 01 extension. The difference is that it sometimes *accepts worse*
parameter sets, in proportion to their probability, so the chains end up
visiting regions in proportion to their posterior probability instead of
converging to a single point ([BUMPS optimizer guide: DREAM](https://bumps.readthedocs.io/en/latest/guide/optimizer.html#dream)).

In [ ]:
np.random.seed(2026)
result = fitter.fit_bayesian(samples=1500, burn=100, thin=1, pop=10)
fitter.plot_results(show_residuals=True, log_scale=True)


## 3 — Posterior data and convergence

`get_posterior()` returns:

- `labels` and a two-dimensional `samples` array;
- 68% and 95% credible intervals;
- separate `chains` where available;
- R-hat diagnostics comparing within-chain and between-chain variation.

R-hat near 1 is desirable. Values above roughly 1.05 are a warning to run
longer, revisit priors, or investigate a multimodal posterior. Trace plots
should look stationary and well mixed rather than drifting or separating.

In [ ]:
posterior = fitter.get_posterior()
print('Parameters:', posterior.labels)
print('Combined sample shape:', posterior.samples.shape)
for name in posterior.labels:
    low, high = posterior.ci_95[name]
    r_hat = posterior.diagnostics.get(name, {}).get('r_hat')
    r_hat_text = f'{r_hat:.3f}' if r_hat is not None else 'n/a'
    print(f'{name:>12}: 95% CI [{low:.6g}, {high:.6g}], R-hat={r_hat_text}')

fitter.plot_trace()


## 4 — Marginal distributions and correlations

A marginal distribution shows uncertainty in one parameter after averaging
over the others. A pair plot and correlation matrix reveal combinations that
the data constrain together. For example, particle contrast, concentration,
and scale can compensate for one another.

**Predict before plotting.** With both SLDs fixed, the plateau height is
$I(0) \propto \mathrm{scale}\cdot V \propto \mathrm{scale}\cdot r^3$ (slide 18).
If the data pin $I(0)$ tightly, which sign of correlation do you expect between
`scale` and `radius`? Is it strong, given that the *positions* of the minima
pin the radius independently (slide 21)? Which parameter should be most
correlated with `background`, and in which q region do they compete?

In [ ]:
fitter.plot_param_distribution('radius')
fitter.plot_posterior_pairs()
fitter.plot_param_correlations()


## 5 — Posterior predictive checking

Parameter intervals are not the final scientific target. A posterior
predictive check propagates sampled parameters through the scattering model.
The resulting band asks whether plausible parameter sets reproduce the
observations across q, and where predictions remain uncertain.

Look especially at the form-factor minima and at the highest q, where the
signal approaches the background. Relate what you see to the lecture's point
that resolution and polydispersity fill in minima (slide 21). If those effects
were present but missing from the model, would the posterior band be wider,
narrower, or simply in the wrong place?

In [ ]:
fitter.plot_posterior_predictive(style='band+draws', n_draws=40)


## 6 — Interpretation

Write a short report addressing:

1. Which parameter is best constrained, and what feature of the curve does it control?
2. Which parameters are correlated, and why?
3. Do the traces and R-hat values support convergence?
4. Where is the predictive band widest?
5. What would you change before publishing these intervals?

In [ ]:
interpretation = (
    'The radius is tightly constrained by the oscillation positions, while scale '
    'and background show stronger trade-offs with intensity normalization. The '
    'R-hat values remain above the convergence target, so this classroom chain '
    'must be lengthened and rechecked before reporting any credible intervals.'
)
print(interpretation)


## Optional export

The raw chain can be analyzed with other tools:

```python
posterior.save_posterior_csv('sphere_posterior.csv')
fitter.save_results('sphere_bayesian_fit.csv')
```

Try changing one prior range or restricting q. Rerun the sampler and explain
whether the posterior changed because of the data or because of the prior.

*Suggested experiment:* restrict q to `0.004–0.025 Å⁻¹`, as in notebook 02,
with `fitter.set_q_range(qmin=0.004, qmax=0.025)`, and rerun. Which posterior
widens most, and does the pair plot now show the Guinier-only degeneracy
between size and scale?

## Further reading

- [BUMPS optimizer guide](https://bumps.readthedocs.io/en/latest/guide/optimizer.html): DREAM, differential
  evolution, and how BUMPS reports uncertainties.
- J. A. Vrugt, *Markov chain Monte Carlo simulation using the DREAM software
  package*, Environ. Model. Softw. 75, 273 (2016): the algorithm behind
  `fit_bayesian`.
- A. Vehtari et al., *Rank-normalization, folding, and localization: an improved
  R-hat for assessing convergence of MCMC*, Bayesian Anal. 16, 667 (2021): why
  R-hat thresholds have tightened from 1.1 to about 1.01.
- [Jacques & Trewhella, *Small-angle scattering for structural biology: expanding the frontier while avoiding the pitfalls*, Protein Sci. 19, 642 (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2867006/): on reporting uncertainty and model ambiguity in SAS results.